# re-admit: Autoresearch Results

This notebook displays the results of autonomous ML experiments for 30-day hospital readmission prediction.

**Dataset**: UCI Diabetes 130-US Hospitals (101,766 encounters, 50 features)

**Best published AUROC**: ~0.70 (CATBoost, PMC 12085305)

---

In [ ]:
# Install dependencies if running in Colab
import subprocess, sys
try:
    import pandas
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'plotly', 'tabulate'])

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

## Load Results

In [ ]:
# Load experiment results
df = pd.read_csv('results.tsv', sep='\t')
df['experiment_num'] = range(1, len(df) + 1)
print(f'Total experiments: {len(df)}')
print(f'Kept: {(df["status"] == "keep").sum()}')
print(f'Discarded: {(df["status"] == "discard").sum()}')
print(f'Crashed: {(df["status"] == "crash").sum()}')
display(df)

## Published Baselines vs Our Results

In [ ]:
# Published baselines
baselines = pd.DataFrame({
    'Model': ['CATBoost (PMC 2025)', 'XGBoost (Liu et al.)', 'LR (Liu et al.)', 'RF (Liu et al.)', 'SVM (2023)', 'LACE Index'],
    'AUROC': [0.700, 0.667, 0.642, 0.630, 0.640, 0.660]
})

best_result = df[df['status'] == 'keep']['auroc'].max() if len(df[df['status'] == 'keep']) > 0 else 0
baselines = pd.concat([baselines, pd.DataFrame({'Model': ['Our Best (Autoresearch)'], 'AUROC': [best_result]})], ignore_index=True)

fig = px.bar(baselines, x='Model', y='AUROC', title='Published Baselines vs Our Autoresearch Results',
             color='AUROC', color_continuous_scale='RdYlGn',
             text='AUROC')
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis_range=[0.5, 0.8], height=500)
fig.show()

## AUROC Progression Over Experiments

In [ ]:
# Plot AUROC over time (only non-crash experiments)
df_valid = df[df['status'] != 'crash'].copy()

fig = go.Figure()

# All experiments
fig.add_trace(go.Scatter(
    x=df_valid['experiment_num'], y=df_valid['auroc'],
    mode='markers+lines', name='All Experiments',
    marker=dict(color=df_valid['status'].map({'keep': 'green', 'discard': 'red'}), size=10),
    text=df_valid['description'], hovertemplate='%{text}<br>AUROC: %{y:.4f}'
))

# Running best
kept = df_valid[df_valid['status'] == 'keep'].copy()
if len(kept) > 0:
    kept['running_best'] = kept['auroc'].cummax()
    fig.add_trace(go.Scatter(
        x=kept['experiment_num'], y=kept['running_best'],
        mode='lines', name='Running Best AUROC',
        line=dict(color='blue', width=3, dash='dash')
    ))

# Published best baseline
fig.add_hline(y=0.700, line_dash='dot', line_color='orange',
              annotation_text='Published Best (0.700)', annotation_position='top left')

fig.update_layout(
    title='Autoresearch AUROC Progression',
    xaxis_title='Experiment Number',
    yaxis_title='AUROC',
    height=500,
    showlegend=True
)
fig.show()

## Experiment Details

In [ ]:
# Show top 10 experiments by AUROC
print('\n=== Top 10 Experiments by AUROC ===')
top10 = df[df['status'] == 'keep'].nlargest(10, 'auroc')[['experiment_num', 'commit', 'auroc', 'f1', 'accuracy', 'description']]
display(top10)

print('\n=== Summary Statistics ===')
kept_df = df[df['status'] == 'keep']
if len(kept_df) > 0:
    print(f'Best AUROC: {kept_df["auroc"].max():.6f}')
    print(f'Mean AUROC (kept): {kept_df["auroc"].mean():.6f}')
    print(f'Best F1: {kept_df["f1"].max():.6f}')
    print(f'Best Accuracy: {kept_df["accuracy"].max():.6f}')
    print(f'Total training time: {df["training_seconds"].sum()/3600:.1f} hours')

## Memory Usage

In [ ]:
df_valid = df[df['status'] != 'crash'].copy()
if len(df_valid) > 0:
    fig = px.scatter(df_valid, x='auroc', y='memory_mb', color='status',
                     hover_data=['description', 'commit'],
                     title='AUROC vs Memory Usage',
                     color_discrete_map={'keep': 'green', 'discard': 'red'})
    fig.update_layout(height=400)
    fig.show()

## Training Time Distribution

In [ ]:
if len(df) > 0:
    fig = px.histogram(df[df['status'] != 'crash'], x='training_seconds', nbins=20,
                       title='Training Time Distribution (seconds)',
                       color='status', color_discrete_map={'keep': 'green', 'discard': 'red'})
    fig.update_layout(height=400)
    fig.show()